# 🎬 3D Gaussian Splatting — FitScan Integration

This notebook generates a 3D Gaussian Splat from a video **sent automatically by the FitScan website**,
and serves the output PLY back to the FitScan dashboard via an ngrok API.

## Two Modes
| Mode | How to use |
|---|---|
| **API Mode** (default) | Run all cells → copy ngrok URL → paste into FitScan `.env` as `GAUSSIAN_COLAB_URL` → upload video on the Scan page |
| **Manual Mode** | Skip the API server cell → run Step M: Manual Upload → run each step manually |

## Pipeline
```
FitScan (video upload)
        ↓  POST /process
  [Colab API Server]
        ↓
  1. Extract frames (ffmpeg)
  2. COLMAP (feature extraction → matching → sparse reconstruction)
  3. Undistort images
  4. 3DGS training (30 000 iterations, ~10 min on T4)
  5. Export point_cloud.ply
        ↓  GET /output.ply
  FitScan Dashboard (3D viewer)
```

## Requirements
- Google Colab **T4 GPU** runtime  
- ~20 GB free disk  
- Video: 10–60 s, 360° walk-around, good lighting, no motion blur

In [ ]:
# ── Step 0: Verify GPU ────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    print('NO GPU detected — please enable: Runtime → Change runtime type → T4 GPU')

In [ ]:
%%bash
# ── Step 1: System dependencies ──────────────────────────────────────────
apt-get update -qq
apt-get install -y -qq --fix-missing \
    ffmpeg colmap git cmake build-essential ninja-build \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev \
    libeigen3-dev libsuitesparse-dev libgflags-dev \
    libgoogle-glog-dev libglew-dev libcgal-dev
echo "--- ffmpeg: $(ffmpeg -version 2>&1 | head -1)"
echo "--- colmap: $(colmap --version 2>&1 || echo ok)"
echo "=== System deps done ==="

In [ ]:
%%bash
# ── Step 2: Python dependencies ──────────────────────────────────────────
pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install -q \
    plyfile tqdm Pillow opencv-python scipy \
    scikit-image matplotlib lpips open3d \
    flask pyngrok requests pycolmap
echo "=== Python deps done ==="

In [ ]:
# ── Step 3: Clone 3D Gaussian Splatting repo ─────────────────────────────
import os

GS_REPO = '/content/gaussian_splatting'

if not os.path.exists(GS_REPO):
    !git clone --depth 1 https://github.com/graphdeco-inria/gaussian-splatting {GS_REPO}
    print('Cloned gaussian-splatting')
else:
    print('Repo already present')

os.chdir(GS_REPO)
!git submodule update --init --recursive
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn
print('Submodules ready')

In [ ]:
# ── Step 4: Define full pipeline as callable functions ───────────────────
# These are used by BOTH the API server and the manual cells below.

import os, shutil, subprocess, json
import pycolmap
import sqlite3
from pathlib import Path

GS_REPO    = '/content/gaussian_splatting'
OUTPUT_PLY = '/content/gs_output.ply'  # final served PLY path


def extract_frames(video_path: str, frames_dir: str,
                   fps: int = 3, max_frames: int = 120) -> list:
    """Extract frames from video using ffmpeg. Returns list of frame paths."""
    os.makedirs(frames_dir, exist_ok=True)
    # Resize to max 1280px wide for faster COLMAP without sacrificing quality
    cmd = (
        f'ffmpeg -i "{video_path}" '
        f'-vf "fps={fps},scale=\'if(gt(iw\,1280)\,1280\,-2)\':ih" '
        f'-q:v 1 -frames:v {max_frames} '
        f'"{frames_dir}/frame_%04d.jpg" -y -loglevel warning'
    )
    os.system(cmd)
    frames = sorted(Path(frames_dir).glob('*.jpg'))
    print(f'[extract_frames] {len(frames)} frames at {frames_dir}')
    return frames


def run_colmap(frames_dir: str, dataset_path: str) -> bool:
    """
    Run COLMAP pipeline: feature extraction → matching → sparse reconstruction.
    Returns True if at least 20 frames were registered.
    """
    dataset_path = Path(dataset_path)
    frames_dir   = Path(frames_dir)
    db_path      = dataset_path / 'database.db'
    sparse_dir   = dataset_path / 'sparse'

    # Clean previous run
    if db_path.exists():    db_path.unlink()
    if sparse_dir.exists(): shutil.rmtree(sparse_dir)
    sparse_dir.mkdir(parents=True, exist_ok=True)

    frames = list(frames_dir.glob('*.jpg'))
    print(f'[colmap] {len(frames)} input frames')

    # 1. Feature extraction
    print('[colmap] 1/3 Feature extraction...')
    ext_opts = pycolmap.FeatureExtractionOptions()
    ext_opts.sift.max_num_features = 8192
    pycolmap.extract_features(
        database_path=db_path,
        image_path=frames_dir,
        camera_mode=pycolmap.CameraMode.SINGLE,
        extraction_options=ext_opts,
        reader_options=pycolmap.ImageReaderOptions(camera_model='OPENCV'),
    )
    con = sqlite3.connect(db_path)
    n_images = con.execute('SELECT COUNT(*) FROM images').fetchone()[0]
    con.close()
    print(f'[colmap]   {n_images} images indexed')

    # 2. Matching — sequential first, exhaustive fallback
    print('[colmap] 2/3 Feature matching...')
    pycolmap.match_sequential(
        database_path=db_path,
        pairing_options=pycolmap.SequentialPairingOptions(
            overlap=20,         # wider overlap for walking-around videos
            loop_detection=False,
        ),
    )
    con = sqlite3.connect(db_path)
    n_pairs = con.execute(
        'SELECT COUNT(*) FROM two_view_geometries WHERE rows > 0'
    ).fetchone()[0]
    con.close()
    print(f'[colmap]   Sequential pairs: {n_pairs}')

    if n_pairs < 30:
        print('[colmap]   Low pairs — adding exhaustive matching...')
        pycolmap.match_exhaustive(database_path=db_path)
        con = sqlite3.connect(db_path)
        n_pairs = con.execute(
            'SELECT COUNT(*) FROM two_view_geometries WHERE rows > 0'
        ).fetchone()[0]
        con.close()
        print(f'[colmap]   Pairs after exhaustive: {n_pairs}')

    # 3. Sparse reconstruction
    print('[colmap] 3/3 Sparse reconstruction...')
    maps = pycolmap.incremental_mapping(
        database_path=db_path,
        image_path=frames_dir,
        output_path=sparse_dir,
    )

    if not maps:
        print('[colmap] FAILED — no model produced')
        return False

    best_key = max(maps, key=lambda k: maps[k].num_reg_images())
    best     = maps[best_key]
    n_reg    = best.num_reg_images()
    pct      = n_reg / max(n_images, 1) * 100
    print(f'[colmap] Registered {n_reg}/{n_images} frames ({pct:.0f}%), '
          f'{best.num_points3D():,} 3D points')

    # Write model to sparse/0
    out = sparse_dir / '0'
    out.mkdir(exist_ok=True)
    best.write_text(str(out))

    if n_reg < 10:
        print('[colmap] WARNING: very few frames registered — results may be poor.')
        print('  Tips: more/better frames, walk slower, avoid textureless backgrounds')
        return n_reg >= 5  # allow through with at least 5

    return True


def run_undistort(dataset_path: str, undistorted_path: str) -> bool:
    """Undistort images (OPENCV → PINHOLE) and move sparse into sparse/0/."""
    import pycolmap
    sparse_0 = Path(dataset_path) / 'sparse' / '0'
    pycolmap.undistort_images(
        output_path=undistorted_path,
        input_path=str(sparse_0),
        image_path=str(Path(dataset_path) / 'input'),
    )

    # Move sparse/*.bin into sparse/0/ as 3DGS expects
    ud_sparse = Path(undistorted_path) / 'sparse'
    ud_sparse_0 = ud_sparse / '0'
    ud_sparse_0.mkdir(exist_ok=True)
    for f in ud_sparse.glob('*.bin'):
        shutil.move(str(f), str(ud_sparse_0 / f.name))

    imgs = list((Path(undistorted_path) / 'images').glob('*'))
    print(f'[undistort] {len(imgs)} undistorted images')
    return len(imgs) > 0


def run_training(dataset_path: str, output_path: str,
                 iterations: int = 30000,
                 log_cb=None) -> bool:
    """Run 3DGS training. Returns True on success."""
    import subprocess, threading

    cmd = [
        'python', 'train.py',
        '-s', dataset_path,
        '-m', output_path,
        '--iterations', str(iterations),
        '--save_iterations', str(iterations),
        '--test_iterations', str(iterations),
        '--densification_interval', '100',
        '--densify_until_iter', str(iterations // 2),
    ]

    proc = subprocess.Popen(
        cmd, cwd=GS_REPO,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )

    for line in proc.stderr:  # training progress goes to stderr
        line = line.rstrip()
        if line:
            print(line)
            if log_cb: log_cb(line)

    proc.wait()
    success = proc.returncode == 0
    print(f'[train] Return code: {proc.returncode}')
    return success


def copy_output_ply(output_path: str) -> str:
    """Copy the trained PLY to OUTPUT_PLY for serving. Returns path or empty string."""
    candidates = sorted(Path(output_path).rglob('point_cloud.ply'),
                        key=lambda p: p.stat().st_size, reverse=True)
    if not candidates:
        print('[export] No point_cloud.ply found')
        return ''
    best = candidates[0]
    shutil.copy(str(best), OUTPUT_PLY)
    size_mb = best.stat().st_size / 1024 / 1024
    print(f'[export] PLY copied: {best} ({size_mb:.1f} MB) → {OUTPUT_PLY}')
    return OUTPUT_PLY


def run_full_pipeline(video_path: str, job_id: str,
                      fps: int = 3, max_frames: int = 120,
                      iterations: int = 30000,
                      status_cb=None) -> bool:
    """
    Run the complete pipeline from video → PLY.
    status_cb(stage, msg) is called at each stage.
    Returns True on success.
    """
    import traceback

    def cb(stage, msg=''):
        print(f'[pipeline:{stage}] {msg}')
        if status_cb: status_cb(stage, msg)

    try:
        base         = f'/content/gs_jobs/{job_id}'
        frames_dir   = f'{base}/input'
        dataset_path = base
        undist_path  = f'{base}_undistorted'
        output_path  = f'{base}_output'
        os.makedirs(base, exist_ok=True)

        cb('extracting', f'fps={fps}, max={max_frames}')
        frames = extract_frames(video_path, frames_dir, fps=fps, max_frames=max_frames)
        if not frames:
            cb('error', 'No frames extracted'); return False

        cb('colmap', f'{len(frames)} frames')
        ok = run_colmap(frames_dir, dataset_path)
        if not ok:
            cb('error', 'COLMAP failed — too few frames registered'); return False

        cb('undistorting')
        ok = run_undistort(dataset_path, undist_path)
        if not ok:
            cb('error', 'Undistortion failed'); return False

        cb('training', f'iterations={iterations}')
        ok = run_training(undist_path, output_path, iterations=iterations)
        if not ok:
            cb('error', 'Training failed'); return False

        cb('exporting')
        ply = copy_output_ply(output_path)
        if not ply:
            cb('error', 'No PLY output found'); return False

        cb('done', f'PLY at {ply}')
        return True

    except Exception as e:
        traceback.print_exc()
        cb('error', str(e))
        return False


print('Pipeline functions defined.')

In [ ]:
# ── Step 5: FitScan API Server ────────────────────────────────────────────
# This cell BLOCKS on purpose — the server must stay running.
# You do NOT need to run any other cells after this one.

# ─────────────────────────────────────────────────────
NGROK_TOKEN  = "YOUR_NGROK_AUTHTOKEN_HERE"   # <-- FILL IN
GS_PORT      = 5002
EXTRACT_FPS  = 3
MAX_FRAMES   = 120
ITERATIONS   = 30000
_OUTPUT_PLY  = '/content/gs_output.ply'      # local to this cell — no dependency on Cell 5
# ─────────────────────────────────────────────────────

import os, uuid, threading, logging, time
from flask import Flask, request, jsonify, send_file
from pyngrok import ngrok

# Free any leftover process on the port
os.system(f'fuser -k {GS_PORT}/tcp 2>/dev/null')
time.sleep(1)

# Kill leftover ngrok tunnels
try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

ngrok.set_auth_token(NGROK_TOKEN)

gs_server = Flask(__name__)
os.makedirs('/content/gs_jobs', exist_ok=True)
logging.getLogger('werkzeug').setLevel(logging.ERROR)

_job = {"id": None, "status": "idle", "stage": "", "message": "", "error": None}
_job_lock = threading.Lock()


@gs_server.route('/', methods=['GET'])
def health():
    return jsonify({"status": "ok", "message": "GS API live", "job": _job['status']})


@gs_server.route('/status', methods=['GET'])
def gs_status():
    output_ready = os.path.exists(_OUTPUT_PLY)
    with _job_lock:
        return jsonify({
            "available":    _job['status'] == 'done' and output_ready,
            "status":       _job['status'],
            "stage":        _job['stage'],
            "message":      _job['message'],
            "error":        _job['error'],
            "job_id":       _job['id'],
            "output_ready": output_ready,
        })


@gs_server.route('/process', methods=['POST'])
def process_video():
    if 'video' not in request.files:
        return jsonify({'error': 'No video file in request'}), 400
    with _job_lock:
        if _job['status'] == 'processing':
            return jsonify({'error': 'Job already running', 'job_id': _job['id']}), 409

    job_id     = uuid.uuid4().hex[:8]
    video_file = request.files['video']
    video_path = f'/content/gs_jobs/upload_{job_id}.mp4'
    video_file.save(video_path)

    fps        = int(request.form.get('fps',        EXTRACT_FPS))
    max_frames = int(request.form.get('max_frames', MAX_FRAMES))
    iterations = int(request.form.get('iterations', ITERATIONS))

    with _job_lock:
        _job.update(id=job_id, status='processing', stage='queued',
                    message='Job queued', error=None)

    def status_cb(stage, msg=''):
        with _job_lock:
            _job['stage']   = stage
            _job['message'] = msg

    def worker():
        try:
            success = run_full_pipeline(
                video_path=video_path, job_id=job_id,
                fps=fps, max_frames=max_frames, iterations=iterations,
                status_cb=status_cb,
            )
        except Exception as e:
            with _job_lock:
                _job['status']  = 'error'
                _job['stage']   = 'error'
                _job['message'] = str(e)
            return
        with _job_lock:
            _job['status']  = 'done' if success else 'error'
            _job['stage']   = 'done' if success else 'error'
            _job['message'] = 'PLY ready at /output.ply' if success else 'Pipeline failed'
        if os.path.exists(video_path):
            os.remove(video_path)

    threading.Thread(target=worker, daemon=True).start()
    return jsonify({'success': True, 'job_id': job_id, 'message': 'Processing started'}), 202


@gs_server.route('/output.ply', methods=['GET'])
def get_ply():
    if not os.path.exists(_OUTPUT_PLY):
        return jsonify({'error': 'No output PLY yet'}), 404
    return send_file(_OUTPUT_PLY, mimetype='application/octet-stream',
                     as_attachment=True, download_name='gaussian_splat.ply')


# ── Start ngrok then run Flask (blocking) ────────────────────────────────
tunnel  = ngrok.connect(GS_PORT)
url_str = tunnel.public_url if hasattr(tunnel, 'public_url') else str(tunnel).split('"')[1]

print('\n' + '='*60)
print('GAUSSIAN SPLATTING API SERVER IS LIVE!')
print(f'  Public URL : {url_str}')
print(f'  Status     : {url_str}/status')
print(f'  Process    : POST {url_str}/process')
print(f'  Output PLY : {url_str}/output.ply')
print('='*60)
print(f'\n>>> Paste into your local FitScan .env file:')
print(f'    GAUSSIAN_COLAB_URL={url_str}')
print('\n>>> Cell stays spinning = server is alive. Do NOT run other cells.')
print('>>> FitScan sends the video automatically when you scan.\n')

gs_server.run(host='0.0.0.0', port=GS_PORT, use_reloader=False, threaded=True)

---
## ✋ Manual Mode (skip if using API)
If you want to run the pipeline manually (not via FitScan), run the cells below.
Otherwise the API server above handles everything automatically.

In [ ]:
# ── Step M: Manual Video Upload ───────────────────────────────────────────
# Only run this cell if NOT using the API server above.

from google.colab import files
import shutil, os

VIDEO_DIR = '/content/input_video'
os.makedirs(VIDEO_DIR, exist_ok=True)

print('Upload your video file (MP4, MOV, AVI)...')
uploaded = files.upload()

video_filename = list(uploaded.keys())[0]
video_path = os.path.join(VIDEO_DIR, video_filename)
shutil.move(video_filename, video_path)
print(f'Video saved: {video_path}')

In [ ]:
# ── Step M1: Extract Frames ───────────────────────────────────────────────
# @param {type:"slider", min:1, max:10, step:1}
EXTRACT_FPS  = 3   # frames per second
MAX_FRAMES   = 120  # maximum frames

FRAMES_DIR = '/content/dataset/input'
frames = extract_frames(video_path, FRAMES_DIR, fps=EXTRACT_FPS, max_frames=MAX_FRAMES)
print(f'Extracted {len(frames)} frames')

# Preview
import matplotlib.pyplot as plt
from PIL import Image

n_preview = min(5, len(frames))
fig, axes = plt.subplots(1, n_preview, figsize=(20, 4))
if n_preview == 1: axes = [axes]
for i, ax in enumerate(axes):
    idx = i * max(1, len(frames) // n_preview)
    ax.imshow(Image.open(frames[idx]))
    ax.axis('off'); ax.set_title(f'Frame {idx}')
plt.suptitle('Sample Extracted Frames')
plt.tight_layout(); plt.show()

In [ ]:
# ── Step M2: COLMAP Pipeline ──────────────────────────────────────────────
DATASET_PATH = '/content/dataset'
ok = run_colmap(FRAMES_DIR, DATASET_PATH)
print('COLMAP OK:', ok)

In [ ]:
# ── Step M3: Undistort Images ─────────────────────────────────────────────
UNDISTORTED_PATH = '/content/dataset_undistorted'
ok = run_undistort(DATASET_PATH, UNDISTORTED_PATH)
print('Undistort OK:', ok)

In [ ]:
# ── Step M4: Train 3D Gaussian Splatting ─────────────────────────────────
# Typical time: ~10 min on T4 at 30 000 iterations
ITERATIONS   = 30000
OUTPUT_PATH  = '/content/output_splat'

ok = run_training(UNDISTORTED_PATH, OUTPUT_PATH, iterations=ITERATIONS)
print('Training OK:', ok)

In [ ]:
# ── Step M5: Export PLY + Make Available via API ──────────────────────────
ply = copy_output_ply(OUTPUT_PATH)

if ply:
    print(f'PLY exported: {ply}')
    print('If the API server is running, the PLY is now served at /output.ply')
    print('FitScan dashboard will automatically pick it up on the next refresh.')
else:
    print('No PLY found — check training output above.')

In [ ]:
# ── Step M6: Visualize with Open3D ───────────────────────────────────────
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ply_path = OUTPUT_PLY if os.path.exists(OUTPUT_PLY) else None
if not ply_path:
    found = list(Path('/content').rglob('point_cloud.ply'))
    ply_path = str(found[0]) if found else None

if ply_path:
    pcd  = o3d.io.read_point_cloud(ply_path)
    pts  = np.asarray(pcd.points)
    cols = np.asarray(pcd.colors) if pcd.has_colors() else None
    print(f'Loaded: {len(pts):,} points from {ply_path}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    views = [(0,2,'X','Z','Top (XZ)'), (0,1,'X','Y','Front (XY)'), (2,1,'Z','Y','Side (ZY)')]
    for ax, (xi, yi, xl, yl, title) in zip(axes, views):
        ax.scatter(pts[:, xi], pts[:, yi],
                   c=cols[:, :3] if cols is not None else 'steelblue',
                   s=0.4, alpha=0.5)
        ax.set(xlabel=xl, ylabel=yl, title=title, aspect='equal', facecolor='#111')
    fig.patch.set_facecolor('#1a1a1a')
    plt.suptitle('Gaussian Splat — Point Cloud Preview', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No PLY file found — run the training step first.')

In [ ]:
# ── Step M7: Download / Save to Drive ────────────────────────────────────
import shutil
from google.colab import files as colab_files

# --- Download ---
if os.path.exists(OUTPUT_PLY):
    print('Downloading PLY file...')
    colab_files.download(OUTPUT_PLY)
else:
    print('No PLY to download — run training first.')

# --- Save to Drive (uncomment if desired) ---
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(OUTPUT_PLY, '/content/drive/MyDrive/gaussian_splat.ply')
# print('Saved to Google Drive')

---
## 🔧 Troubleshooting

| Problem | Fix |
|---|---|
| `ngrok` auth error | Create free account at ngrok.com → Dashboard → Auth Token |
| COLMAP registers < 20% frames | Walk slower around subject; 3–4 m distance; avoid plain walls |
| Training OOM | Reduce `MAX_FRAMES` to 60 or lower `ITERATIONS` to 15000 |
| Blurry splat | Ensure video has no motion blur; increase FPS to 5 |
| API returns 409 | Previous job still running — check `/status` |
| Dashboard shows "Point Cloud" not "Gaussian Splat" | Gaussian Colab URL not set — add `GAUSSIAN_COLAB_URL=...` to `.env` and restart |

## 📚 Viewers
- **Web**: https://playcanvas.com/supersplat/editor (drag & drop `.ply`)
- **Web**: https://antimatter15.com/splat/
- **Desktop**: Download & run `python render.py` from the 3DGS repo